# Crawl4AI vs Scrapy vs BeautifulSoup

Concise notebook for a YouTube coding demo. Focus: Crawl4AI delivers markdown + metadata for AI with minimal code.


**Notebook map**

- Compare three Python scrapers on the same URL.
- Keep implementation compact (<250 lines) and high level.
- Spotlight Crawl4AI's markdown output and AI-readiness.
- Cells assume dependencies + network are available.


**Imports and test target**
Swap `TEST_URL` for whatever page you want to show.


In [13]:
import asyncio
import time
from typing import Any, Dict

import requests
from bs4 import BeautifulSoup
from crawl4ai import AsyncWebCrawler
import scrapy
from scrapy.crawler import CrawlerProcess

TEST_URL = "https://www.python.org"


**Utility**


In [19]:
def trim(text: str, length: int = 500) -> str:
    return text if len(text) <= length else text[:length] + "..."


## BeautifulSoup baseline

Traditional scraping: manual HTTP request, manual parsing, manual cleanup.


In [15]:
def scrape_with_bs(url: str) -> Dict[str, Any]:
    start = time.time()
    response = requests.get(url, timeout=10)
    response.raise_for_status()
    soup = BeautifulSoup(response.content, "html.parser")
    for tag in soup(["script", "style"]):
        tag.decompose()
    text = " ".join(soup.get_text(separator=" ").split())
    duration = time.time() - start
    return {
        "tool": "BeautifulSoup",
        "title": soup.title.string if soup.title else None,
        "preview": trim(text),
        "markdown": False,
        "links": "Manual extraction",
        "media": "Manual extraction",
        "duration": duration,
        "ai_ready": False,
        "notes": "Imperative parsing; no markdown; needs selectors.",
    }


## Scrapy skeleton

Powerful crawler with async engine, but needs spiders, settings, and pipelines before you get usable output.


In [16]:
def scrape_with_scrapy(url: str) -> Dict[str, Any]:
    captured: Dict[str, Any] = {}
    start = time.time()

    class MiniSpider(scrapy.Spider):
        name = "mini"
        start_urls = [url]

        def parse(self, response):
            text = " ".join(response.css("body ::text").getall())
            captured["title"] = response.css("title::text").get()
            captured["preview"] = trim(text)

    process = CrawlerProcess(settings={"LOG_ENABLED": False})
    process.crawl(MiniSpider)
    process.start()
    duration = time.time() - start
    return {
        "tool": "Scrapy",
        "title": captured.get("title"),
        "preview": captured.get("preview", ""),
        "markdown": False,
        "links": "Selectors needed",
        "media": "Pipelines needed",
        "duration": duration,
        "ai_ready": False,
        "notes": "Async engine, but still manual extraction + settings.",
    }


## Crawl4AI one-liner

AI-first crawler that returns clean markdown, links, media, and metadata in a single call.


In [17]:
async def scrape_with_crawl4ai(url: str) -> Dict[str, Any]:
    start = time.time()
    async with AsyncWebCrawler(verbose=False) as crawler:
        result = await crawler.arun(url=url)
    duration = time.time() - start
    return {
        "tool": "Crawl4AI",
        "title": result.metadata.get("title"),
        "preview": trim(result.markdown),
        "markdown": True,
        "links": {"internal": len(result.links.get("internal", [])), "external": len(result.links.get("external", []))},
        "media": {
            "images": len(result.media.get("images", [])),
            "videos": len(result.media.get("videos", [])),
            "audios": len(result.media.get("audios", [])),
        },
        "duration": duration,
        "ai_ready": True,
        "notes": "One async call returns markdown + metadata for LLMs.",
    }


## Comparison helper

Runs all three approaches and prints a compact feature table.


In [18]:
async def compare(url: str = TEST_URL):
    bs = scrape_with_bs(url)
    scrapy_result = scrape_with_scrapy(url)
    crawl4ai_result = await scrape_with_crawl4ai(url)
    ordered = [bs, scrapy_result, crawl4ai_result]

    header = f"{'Tool':<15}{'Markdown':<12}{'AI Ready':<12}{'Duration(s)':<12}{'Links/Media':<18}Notes"
    print(header)
    print("-" * len(header))
    for result in ordered:
        lm = result["links"] if isinstance(result["links"], str) else f"{result['links']} / {result['media']}"
        print(
            f"{result['tool']:<15}{str(result['markdown']):<12}{str(result['ai_ready']):<12}{result['duration']:<12.2f}{str(lm):<18}{result['notes']}"
        )

    return ordered


# Uncomment to run live (needs network + crawl4ai installed)
# asyncio.run(compare(TEST_URL))
